# NB09 — Vest Detection Boost (fine-tune with new vest data)

**Problem:** Vest detection on novel real-world images is weak. The original dataset has only ~2.2% `vest` / ~2.9% `no-vest` boxes, with limited visual diversity (colors, styles, contexts).

**New data:** [Safety Vests dataset](https://universe.roboflow.com/roboflow-universe-projects/safety-vests) (Roboflow Universe, **CC BY 4.0**, mirrored on [Kaggle](https://www.kaggle.com/datasets/adilshamim8/safety-vests-detection-dataset)) — 3,897 images with `Safety Vest` / `NO-Safety Vest` boxes. Verified color variety: orange, lime, yellow, green, red vests across construction, warehouse, utility, and industrial scenes.

**Strategy:**
1. Convert CSV annotations → YOLO format, map to our class ids (`vest`=2, `no-vest`=3).
2. **Train-only merge** — new images go into training only; original val/test stay untouched so metrics remain comparable to NB05/NB07.
3. Perceptual-hash dedup — drop new images near-duplicating val/test (leakage guard, Hamming ≤ 5) or train (Hamming ≤ 2).
4. Fine-tune from the deployed YOLO11s weights (40 epochs, low LR) rather than retraining from scratch.
5. Evaluate per-class AP on the ORIGINAL test set vs the NB07 baseline, then export ONNX for deployment.

**Baseline to beat (NB07, test set):** mAP50 93.2% overall — vest 92.0%, no-vest 92.1% (in-distribution; real-world vest recall is the actual weakness).

In [ ]:
import os
from pathlib import Path

import torch

os.environ["YOLO_AUTOINSTALL"] = "false"  # stop ultralytics installing onnxruntime-gpu/numpy2 into the venv

ROOT = Path(r"c:\Users\nwagb\Desktop\SponsorshipGlobalTalentPrep\PPE_Compliance_detection")
KAGGLE_SRC = Path.home() / ".cache/kagglehub/datasets/adilshamim8/safety-vests-detection-dataset/versions/1"
VEST_IMG = ROOT / "data/vest_boost/images"
VEST_LBL = ROOT / "data/vest_boost/labels"
DATA_V2 = ROOT / "data/ppe_dataset_v2.yaml"
BASE_WEIGHTS = ROOT / "results/weights/ppe_yolo11s.pt"
CLASS_NAMES = ["hardhat", "no-hardhat", "vest", "no-vest", "person"]

print("CUDA:", torch.cuda.is_available(), "-", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
print("base weights:", BASE_WEIGHTS.exists(), "| data v2 yaml:", DATA_V2.exists())

## 1. Data preparation

Idempotent: skips if `data/vest_boost/` is already populated. Downloads via `kagglehub` (anonymous, no API key needed for public datasets) if the Kaggle cache is missing.

In [ ]:
import csv
from collections import defaultdict

import numpy as np
from PIL import Image
import imagehash

CLASS_MAP = {"Safety Vest": 2, "NO-Safety Vest": 3}


def hash_dir(d):
    hs = []
    for f in sorted(os.listdir(d)):
        try:
            hs.append(imagehash.phash(Image.open(Path(d) / f)))
        except Exception:
            pass
    return np.array([h.hash.flatten() for h in hs], dtype=bool)


def min_dist(h, guard):
    return int(np.min(np.count_nonzero(guard != h, axis=1))) if len(guard) else 64


if any(VEST_IMG.glob("*.jpg")):
    n = len(list(VEST_IMG.glob("*.jpg")))
    print(f"vest_boost already prepared: {n} images — skipping conversion")
else:
    if not KAGGLE_SRC.exists():
        import kagglehub
        KAGGLE_SRC = Path(kagglehub.dataset_download("adilshamim8/safety-vests-detection-dataset"))
    VEST_IMG.mkdir(parents=True, exist_ok=True)
    VEST_LBL.mkdir(parents=True, exist_ok=True)

    print("hashing existing splits (leakage guard)...")
    guard_valtest = np.concatenate([
        hash_dir(ROOT / "data/processed/images/val"),
        hash_dir(ROOT / "data/processed/images/test"),
    ])
    guard_train = hash_dir(ROOT / "data/processed/images/train")

    rows_by_image = defaultdict(list)
    for split in ["train", "valid", "test"]:
        with open(KAGGLE_SRC / split / split / "_annotations.csv", newline="") as f:
            for row in csv.DictReader(f):
                rows_by_image[(split, row["filename"])].append(row)

    kept = skipped_dup = skipped_bad = 0
    box_counts = {2: 0, 3: 0}
    for (split, fname), rows in rows_by_image.items():
        src_img = KAGGLE_SRC / split / split / fname
        try:
            im = Image.open(src_img)
            im.verify()
            im = Image.open(src_img)
            w, h = im.size
            new_hash = np.array(imagehash.phash(im).hash.flatten(), dtype=bool)
        except Exception:
            skipped_bad += 1
            continue
        if min_dist(new_hash, guard_valtest) <= 5 or min_dist(new_hash, guard_train) <= 2:
            skipped_dup += 1
            continue
        lines = []
        for r in rows:
            cls = CLASS_MAP.get(r["class"])
            if cls is None:
                continue
            x1, y1, x2, y2 = (float(r[k]) for k in ("xmin", "ymin", "xmax", "ymax"))
            x1, x2 = max(0, min(x1, x2)), min(w, max(x1, x2))
            y1, y2 = max(0, min(y1, y2)), min(h, max(y1, y2))
            bw, bh = x2 - x1, y2 - y1
            if bw < 2 or bh < 2:
                continue
            lines.append(f"{cls} {(x1 + x2) / 2 / w:.6f} {(y1 + y2) / 2 / h:.6f} {bw / w:.6f} {bh / h:.6f}")
            box_counts[cls] += 1
        if not lines:
            skipped_bad += 1
            continue
        stem = Path(f"sv_{split}_{fname}").stem
        Image.open(src_img).convert("RGB").save(VEST_IMG / f"{stem}.jpg", quality=95)
        (VEST_LBL / f"{stem}.txt").write_text("\n".join(lines))
        kept += 1

    print(f"kept: {kept} | near-dup skipped: {skipped_dup} | bad/empty skipped: {skipped_bad}")
    print(f"new boxes -> vest: {box_counts[2]}, no-vest: {box_counts[3]}")

if not DATA_V2.exists():
    DATA_V2.write_text(
        f"""# v2: original dataset + vest_boost (Safety Vests, CC BY 4.0, train-only)\npath: {ROOT / 'data'}\ntrain:\n  - processed/images/train\n  - vest_boost/images\nval: processed/images/val\ntest: processed/images/test\nnc: 5\nnames:\n- hardhat\n- no-hardhat\n- vest\n- no-vest\n- person\n"""
    )
    print("wrote", DATA_V2)

In [ ]:
# Sanity check: draw boxes on a few converted samples
import random

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

samples = random.sample(sorted(VEST_IMG.glob("*.jpg")), 6)
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, img_path in zip(axes.flat, samples):
    im = Image.open(img_path)
    w, h = im.size
    ax.imshow(im)
    for line in (VEST_LBL / f"{img_path.stem}.txt").read_text().splitlines():
        c, cx, cy, bw, bh = line.split()
        c, cx, cy, bw, bh = int(c), float(cx) * w, float(cy) * h, float(bw) * w, float(bh) * h
        color = "lime" if c == 2 else "red"
        ax.add_patch(mpatches.Rectangle((cx - bw / 2, cy - bh / 2), bw, bh, fill=False, color=color, lw=2))
    ax.set_title(img_path.name[:40], fontsize=8)
    ax.axis("off")
plt.suptitle("vest_boost samples — green=vest, red=no-vest")
plt.tight_layout()
plt.show()

## 2. Fine-tune YOLO11s from the deployed weights

40 epochs at low LR (0.002 vs 0.01 from scratch), same 1280px/batch=4 regime as NB05. ~45-80 min on the RTX 4060. `torch.cuda.empty_cache()` first — 8 GB VRAM is tight at 1280.

In [ ]:
from ultralytics import YOLO

torch.cuda.empty_cache()

model = YOLO(BASE_WEIGHTS)
results = model.train(
    data=str(DATA_V2),
    epochs=40,
    imgsz=1280,
    batch=4,
    optimizer="SGD",
    lr0=0.002,
    lrf=0.05,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=1,
    cos_lr=True,
    box=7.5,
    cls=0.5,
    dfl=1.5,
    patience=15,
    seed=42,
    device=0,
    workers=2,
    name="ppe_yolo11s_vboost",
    verbose=True,
)

## 3. Evaluate on the ORIGINAL test set

Auto-detects the newest `ppe_yolo11s_vboost*` run folder (Ultralytics suffixes re-runs). Compares per-class AP50 against the NB07 baseline.

In [ ]:
import pandas as pd

run_dirs = sorted(Path("runs/detect").glob("ppe_yolo11s_vboost*"), key=os.path.getmtime)
assert run_dirs, "no vboost run found — did training complete?"
best = run_dirs[-1] / "weights" / "best.pt"
print("evaluating:", best)

tuned = YOLO(best)
metrics = tuned.val(data=str(DATA_V2), split="test", imgsz=1280, batch=4, verbose=False)

BASELINE_AP50 = {"hardhat": 0.940, "no-hardhat": 0.966, "vest": 0.920, "no-vest": 0.921, "person": 0.915}
rows = []
for i, name in enumerate(CLASS_NAMES):
    ap50 = float(metrics.box.ap50[i])
    rows.append({"class": name, "baseline AP50": BASELINE_AP50[name], "vboost AP50": round(ap50, 3), "delta": round(ap50 - BASELINE_AP50[name], 3)})
df = pd.DataFrame(rows)
print(df.to_string(index=False))
print(f"\noverall mAP50: {metrics.box.map50:.3f} (baseline 0.932) | mAP50-95: {metrics.box.map:.3f} (baseline 0.639)")
df.to_csv(ROOT / "results" / "nb09_vboost_comparison.csv", index=False)

In [ ]:
# Real-world spot check: old model vs fine-tuned on the deployment examples
# (add your own problem images to this list — that's the real test)
spot_images = sorted((ROOT / "deployment" / "examples").glob("*.jpg"))

old = YOLO(BASE_WEIGHTS)
fig, axes = plt.subplots(len(spot_images), 2, figsize=(14, 5 * len(spot_images)))
for row, img in enumerate(spot_images):
    for col, (m, label) in enumerate([(old, "baseline"), (tuned, "vboost")]):
        r = m.predict(Image.open(img), imgsz=640, conf=0.25, verbose=False)[0]
        axes[row][col].imshow(r.plot()[..., ::-1])
        vests = sum(1 for b in r.boxes if int(b.cls) in (2, 3))
        axes[row][col].set_title(f"{img.name} — {label} ({len(r.boxes)} boxes, {vests} vest-class)", fontsize=9)
        axes[row][col].axis("off")
plt.tight_layout()
plt.show()

## 4. Export ONNX and stage for deployment

Only run once you're happy with section 3. Overwrites `deployment/ppe_yolo11s.onnx` and copies the new `.pt` to `results/weights/`. Deploy by copying the ONNX into the `PPE/` HF Space clone and pushing.

In [ ]:
import shutil

onnx_path = tuned.export(format="onnx", imgsz=640, opset=12, simplify=True)
shutil.copy(onnx_path, ROOT / "deployment" / "ppe_yolo11s.onnx")
shutil.copy(best, ROOT / "results" / "weights" / "ppe_yolo11s_vboost.pt")
print("staged deployment/ppe_yolo11s.onnx")
print("\nTo deploy:")
print("  cp deployment/ppe_yolo11s.onnx PPE/ppe_yolo11s.onnx")
print("  cd PPE && git add -A && git commit -m 'model: fine-tuned with vest_boost data' && git push")

## Notes

- **Attribution (CC BY 4.0):** credit "Safety Vests dataset, Roboflow Universe" in the README when this ships.
- If vest AP on the original test set drops a point while real-world detection improves, that's an acceptable trade — the original test vests are narrow in style, and the goal is generalization.
- **Further variety (optional):** [SH17](https://github.com/ahmadmughees/SH17dataset) (8,099 Pexels images, `safety-vest` + `person`, CC BY-NC-SA — non-commercial) would add high-res manufacturing scenes; it lacks a `no-vest` class so it only boosts `vest`.
- If VRAM overflows at 1280/batch=4 during fine-tune, drop to batch=3 (don't reduce imgsz — small vests need the resolution).